# Chạy một thí nghiệm CURE trên Kaggle

Chia việc theo (benchmark, tập OOD), mỗi phần một tiến trình. Kết quả lưu theo từng luồng nên chạy lại sẽ tiếp từ chỗ dừng;
gắn Output của lần trước làm Input để tiếp tục. Đầu ra: `/kaggle/working/<thí nghiệm>_results.zip`.

In [ ]:
EXPERIMENT = "robustness"
WORKERS = 4

In [ ]:
print("@@CELL 1@@", flush=True)
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
T0 = time.time()
def tick(m): print(f"[{(time.time() - T0) / 60:6.1f} phút] {m}", flush=True)

code_dir = next(t.parent for t in sorted(INPUT.rglob("pyproject.toml")) if 'name = "cure-ood"' in t.read_text(encoding="utf-8"))
REPO = WORK / "CURE"
if not REPO.exists():
    shutil.copytree(code_dir, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))  # cài dạng -e ko có hiệu lực ngay trong tiến trình đang chạy
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], capture_output=True, text=True)
print("faiss:", "ok" if r.returncode == 0 else "bỏ qua: " + r.stderr[-200:])

FEAT = WORK / "features"; FEAT.mkdir(exist_ok=True)
for f in INPUT.rglob("*.np[yz]"):
    if f.suffix == ".npz" or f.name.startswith(("feats_", "labels_")):
        if not (FEAT / f.name).exists():
            (FEAT / f.name).symlink_to(f)
os.environ.update(CURE_FEATURES=str(FEAT), CURE_HF_AUTO="0")
os.chdir(REPO)
print(open("VERSION.txt").read(), sorted(p.name for p in FEAT.glob("*.npz")))

prev = next((p for p in INPUT.rglob(f"{EXPERIMENT}_results.zip")), None)
if prev:
    with zipfile.ZipFile(prev) as z:
        z.extractall(REPO)
    print("nạp lại:", len(list((REPO / "results" / EXPERIMENT).glob("*.jsonl"))), "file")

In [ ]:
print("@@CELL 2@@", flush=True)
import yaml
from cure_ood.data import Benchmark
from cure_ood.eval.runner import Experiment
exp = Experiment.load(EXPERIMENT)
jobs = []
for spec in exp.benchmark_specs():
    b = Benchmark.load(spec)
    name = spec if isinstance(spec, str) else spec["name"]
    for dom in exp.domains or b.domains:
        jobs.append((f"{name}/{dom}", [sys.executable, "-m", "cure_ood.cli", "run", EXPERIMENT,
                                       "--benchmark", name, "--domain", dom]))
print(len(jobs), "phần việc:", [j[0] for j in jobs])

In [ ]:
print("@@CELL 3@@", flush=True)
ENV = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
LOGS = REPO / "results/logs"; LOGS.mkdir(parents=True, exist_ok=True)

def run_all(jobs, workers, tag="job"):
    pend, run, failed = list(enumerate(jobs)), {}, []
    while pend or run:
        while pend and len(run) < workers:
            i, (name, cmd) = pend.pop(0)
            log = open(LOGS / f"{tag}_{i}.log", "w")
            run[i] = (subprocess.Popen(cmd, env=ENV, stdout=log, stderr=subprocess.STDOUT), log, name)
        for i, (p, log, name) in list(run.items()):
            if p.poll() is not None:
                log.close(); run.pop(i)
                if p.returncode:
                    failed.append((name, jobs[i][1]))
                tick(f"xong {name} (mã {p.returncode}); còn {len(pend) + len(run)}")
        time.sleep(30)
    return failed

failed = run_all(jobs, WORKERS)
if failed:
    tick(f"thử lại {len(failed)} phần lỗi")
    failed = run_all([(n, c) for n, c in failed], max(1, WORKERS // 2), "retry")
tick(f"xong tất cả; còn lỗi: {[n for n, _ in failed]}")

In [ ]:
print("@@CELL 4@@", flush=True)
subprocess.run([sys.executable, "-m", "cure_ood.cli", "report", EXPERIMENT], env=ENV, check=False)
out = WORK / f"{EXPERIMENT}_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (REPO / "results").rglob("*"):
        if f.is_file() and (EXPERIMENT in f.parts or "logs" in f.parts):
            z.write(f, f.relative_to(REPO))
print(out, round(out.stat().st_size / 1e6, 1), "MB")